# 11 — Common-Support Market Comparison

This notebook compares the locked weather-derived event probabilities with
Polymarket prices on exact common support.

The probabilistic model, continuous dispersion scale and uniform probability
mixing parameter were selected before market prices and evaluation outcomes
were introduced.

This notebook evaluates predictive probabilities. It does not select a
trading strategy and does not calculate trading returns.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        required = (
            candidate
            / "data/manifests/"
            "11_market_comparison_manifest.json"
        )

        if required.exists():
            return candidate

    raise FileNotFoundError(
        "Repository root not found."
    )


ROOT = locate_repository(Path.cwd())

manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "11_market_comparison_manifest.json"
    ).read_text(encoding="utf-8")
)

alias_audit = json.loads(
    (
        ROOT
        / "outputs/diagnostics/"
        "11_market_alias_difference_audit.json"
    ).read_text(encoding="utf-8")
)

probabilities = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "11_common_support_probability_panel.csv"
)

book_scores = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "11_common_support_book_score_panel.csv"
)

date_scores = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "11_common_support_date_score_panel.csv"
)

block_summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "11_common_support_block_summary.csv"
)

missing_books = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "11_common_support_missing_books.csv"
)

integrity = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "11_common_support_integrity_checks.csv"
)

print("Status:", manifest["status"])
print(
    "Canonical source:",
    manifest["canonical_market_source"],
)
print(
    "Market price field:",
    manifest["market_price_column"],
)
print(
    "Common-support books:",
    manifest["common_support_probability_books"],
)
print(
    "Common-support dates:",
    manifest["common_support_dates"],
)

Status: COMMON_SUPPORT_MARKET_COMPARISON_COMPLETE
Canonical source: data/processed/18sA_canonical_source_adapters/18sA_canonical_market_panel.csv
Market price field: p_market
Common-support books: 154
Common-support dates: 40


## Market-source decision

The canonical `18sA` source adapter is the authoritative historical market
panel. The expanded scoring panel is a later derived copy and is retained only
as diagnostic evidence.

Exact equality between these files is not imposed. The canonical source takes
precedence whenever their contents differ.

In [2]:
assert manifest["market_source_certified"] is True
assert manifest["canonical_source_precedence"] is True

assert (
    manifest["exact_alias_equivalence_required"]
    is False
)

assert (
    manifest["market_price_column"]
    == "p_market"
)

print(
    "Canonical source independently certified:",
    alias_audit["canonical_source_certified"],
)

print(
    "Exact alias equivalence:",
    alias_audit["exact_alias_equivalence"],
)

print(
    "Exact alias equivalence required:",
    alias_audit["exact_alias_equivalence_required"],
)

Canonical source independently certified: True
Exact alias equivalence: True
Exact alias equivalence required: False


## Market probability normalisation

The eleven contracts trade separately, so their raw prices need not sum
exactly to one.

For categorical evaluation, the raw market prices are normalised within each
date and decision-rule book:

\[
\widetilde p^{\mathrm{mkt}}_{d,r,j}
=
\frac{p^{\mathrm{mkt}}_{d,r,j}}
{\sum_{\ell=1}^{11}p^{\mathrm{mkt}}_{d,r,\ell}}.
\]

The raw prices remain unchanged for the later trading analysis.

In [3]:
book_sizes = probabilities.groupby(
    "row_id"
).size()

model_sums = probabilities.groupby(
    "row_id"
)["model_probability"].sum()

market_sums = probabilities.groupby(
    "row_id"
)["market_probability_normalised"].sum()

assert book_sizes.eq(11).all()

assert np.allclose(
    model_sums.to_numpy(dtype=float),
    1.0,
    atol=1.0e-10,
    rtol=0.0,
)

assert np.allclose(
    market_sums.to_numpy(dtype=float),
    1.0,
    atol=1.0e-10,
    rtol=0.0,
)

print(
    "Every scored book contains eleven events:",
    True,
)

print(
    "Model probability books sum to one:",
    True,
)

print(
    "Normalised market books sum to one:",
    True,
)

Every scored book contains eleven events: True
Model probability books sum to one: True
Normalised market books sum to one: True


## Exact common support

Only complete event books observed in both sources enter the comparison.
Unpaired books are reported explicitly and are not imputed.

In [4]:
print(
    missing_books.sort_values(
        [
            "target_date",
            "decision_rule",
            "support_status",
        ]
    ).to_string(index=False)
)

model_only = int(
    missing_books[
        "support_status"
    ].eq("model_only").sum()
)

market_only = int(
    missing_books[
        "support_status"
    ].eq("market_only").sum()
)

assert model_only == manifest["model_only_books"]
assert market_only == manifest["market_only_books"]

print()
print("Model-only books:", model_only)
print("Market-only books:", market_only)

            book_key target_date decision_rule chronology_block support_status                            reason
2026-06-06|12h_prior  2026-06-06     12h_prior    external_test     model_only no complete canonical market book
2026-06-06|24h_prior  2026-06-06     24h_prior    external_test     model_only no complete canonical market book
2026-06-07|12h_prior  2026-06-07     12h_prior    external_test     model_only no complete canonical market book
2026-06-07|24h_prior  2026-06-07     24h_prior    external_test     model_only no complete canonical market book
2026-06-08|24h_prior  2026-06-08     24h_prior    external_test     model_only no complete canonical market book
 2026-06-24|6h_prior  2026-06-24      6h_prior              NaN    market_only   no locked model prediction book

Model-only books: 5
Market-only books: 1


## Paired categorical evaluation

For the realised event \(J_d\), the categorical log score is

\[
-\log p_{d,r,J_d}.
\]

The multiclass Brier score is

\[
\sum_{j=1}^{11}
\left(
p_{d,r,j}
-
\mathbf{1}_{\{J_d=j\}}
\right)^2.
\]

Smaller values are better. The reported difference is model score minus
market score, so a negative value favours the weather-derived model.

The four decision rules are averaged within each settlement date before block
means are calculated. Settlement date is the uncertainty unit.

In [5]:
display_columns = [
    "chronology_block",
    "dates",
    "probability_books",
    "decision_rules",
    "mean_date_model_log_score",
    "mean_date_market_log_score",
    "mean_date_log_score_difference_model_minus_market",
    "standard_error_date_log_score_difference_model_minus_market",
    "model_log_score_date_win_share",
    "mean_date_model_brier_score",
    "mean_date_market_brier_score",
    "mean_date_brier_score_difference_model_minus_market",
    "standard_error_date_brier_score_difference_model_minus_market",
    "model_brier_score_date_win_share",
]

print(
    block_summary[
        display_columns
    ].to_string(index=False)
)

assert set(
    block_summary["chronology_block"]
) == {
    "holdout",
    "external_test",
}

score_columns = [
    "model_categorical_log_score",
    "market_categorical_log_score",
    "model_multiclass_brier_score",
    "market_multiclass_brier_score",
]

assert np.isfinite(
    book_scores[
        score_columns
    ].to_numpy(dtype=float)
).all()

chronology_block  dates  probability_books  decision_rules  mean_date_model_log_score  mean_date_market_log_score  mean_date_log_score_difference_model_minus_market  standard_error_date_log_score_difference_model_minus_market  model_log_score_date_win_share  mean_date_model_brier_score  mean_date_market_brier_score  mean_date_brier_score_difference_model_minus_market  standard_error_date_brier_score_difference_model_minus_market  model_brier_score_date_win_share
   external_test     30                114               4                   1.550654                    1.260923                                           0.289732                                                     0.088807                             0.2                     0.720809                      0.648471                                             0.072339                                                       0.028597                          0.233333
         holdout     10                 40               4        

## Evidential interpretation

The June external block favours the market under both categorical scores.

The smaller holdout comparison is closer. The market records the lower mean
categorical log score, while the mean multiclass Brier scores are approximately
equal.

These results do not alter the model or calibration parameters selected from
development data.

In [6]:
passed = (
    integrity["passed"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({"true", "1"})
)

assert passed.all()

assert (
    manifest["market_prices_used_for_model_selection"]
    is False
)

assert manifest["model_reselected"] is False

assert (
    manifest["continuous_calibration_reselected"]
    is False
)

assert (
    manifest["probability_calibration_reselected"]
    is False
)

assert (
    manifest["trading_strategy_selected"]
    is False
)

assert (
    manifest["trading_returns_calculated"]
    is False
)

print(
    "All Notebook 11 integrity checks passed:",
    True,
)

print(
    "Market prices used for model selection:",
    False,
)

print(
    "Trading returns calculated:",
    False,
)

All Notebook 11 integrity checks passed: True
Market prices used for model selection: False
Trading returns calculated: False
